# Intelligent Cyberbullying Detection Framework
## Module: Exploratory Dataset Analysis (Person 1 - Dataset Collection & Preprocessing)

### Overview
This notebook provides an in-depth exploratory data analysis (EDA) of **5 benchmark cyberbullying and online toxicity datasets** collected from public academic and repository sources (Kaggle, Hugging Face, GitHub).

### Research Objective
Examine the structure, labeling schemes, class distributions, missing values, duplicates, and compatibility of each dataset to harmonize them into a canonical 5-class cyberbullying taxonomy:
1. **Not Cyberbullying** (Class 0)
2. **Insult** (Class 1)
3. **Threat** (Class 2)
4. **Hate Speech** (Class 3)
5. **Harassment** (Class 4)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.append(os.path.abspath('../src'))
from dataset_loader import DatasetLoader, DATASET_METADATA
from label_mapping import LabelMapper, LABEL_MAP, LABEL_DEFINITIONS

sns.set_theme(style='whitegrid')
print('Environment and modules loaded successfully!')

## 1. Dataset Collection & Metadata
We inspect the provenance, platform domain, license, and raw schema for the 5 benchmark datasets.

In [ ]:
loader = DatasetLoader()
summary_df = loader.inspect_dataset_summary()
summary_df[['Dataset Key', 'Dataset Name', 'Platform', 'License', 'Total Rows', 'Missing Texts', 'Duplicate Texts']]

## 2. In-Depth Analysis of Individual Datasets
### 2.1 Davidson et al. (2017) Hate Speech & Offensive Language
- **Source**: Davidson et al. (ICWSM 2017), Twitter corpus.
- **Classes**: 0: Hate Speech, 1: Offensive Language, 2: Neither.

In [ ]:
df_dav = loader.load_davidson()
print(f'Total records: {len(df_dav):,}')
print('Raw class distribution:')
print(df_dav['class'].value_counts())
df_dav[['tweet', 'class']].head(4)

### 2.2 Wang et al. (2021) Cyberbullying Tweets
- **Source**: Kaggle / IEEE (Wang et al., 2021), Twitter corpus.
- **Classes**: `not_cyberbullying`, `religion`, `age`, `gender`, `ethnicity`, `other_cyberbullying`.

In [ ]:
df_cb = loader.load_cyberbullying_tweets()
print(f'Total records: {len(df_cb):,}')
print('Raw class distribution:')
print(df_cb['cyberbullying_type'].value_counts())
df_cb.head(4)

### 2.3 Jigsaw Toxic Comment Classification Challenge
- **Source**: Google Conversation AI / Jigsaw (2018), Wikipedia Talk Pages.
- **Multi-label flags**: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`.

In [ ]:
df_jig = loader.load_jigsaw()
print(f'Total records: {len(df_jig):,}')
cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print('Toxicity flag occurrences:')
print(df_jig[cols].sum())
df_jig[['comment_text'] + cols].head(4)

### 2.4 TRAC-1 Aggression Identification
- **Source**: TRAC@COLING 2018 Shared Task (Kumar et al.), Facebook user comments.
- **Classes**: `NAG` (Non-Aggressive), `CAG` (Covertly Aggressive), `OAG` (Overtly Aggressive).

In [ ]:
df_trac = loader.load_trac1()
print(f'Total records: {len(df_trac):,}')
print('Raw class distribution:')
print(df_trac['label'].value_counts())
df_trac.head(4)

### 2.5 Formspring.me Cyberbullying Dataset & Limitation Documentation
- **Source**: Reynolds et al. (ICMLA 2011), Formspring.me Q&A posts.
- **Limitation Analysis**: Formspring provides binary crowd annotations (`bully1..3`, `ans1..3`), but lacks the semantic granularity to classify into Insult, Threat, Hate Speech, or Harassment. Forcing binary labels into the 5 classes would violate the project's strict requirement against fabricating or forcing ambiguous labels.

In [ ]:
df_fs = loader.load_formspring()
limitation_doc = LabelMapper.inspect_formspring_limitation(df_fs)
print('Formspring Limitation Documented:')
for k, v in limitation_doc.items():
    print(f'  {k}: {v}')

## 3. Label Harmonization & Cross-Dataset Integration
We inspect the harmonized mapping results across the 4 compatible datasets.

In [ ]:
from dataset_integration import DatasetIntegrator
integrator = DatasetIntegrator()
df_combined, raw_breakdown = integrator.load_and_map_all()

plt.figure(figsize=(10, 5))
sns.countplot(data=df_combined, x='label_name', hue='original_dataset')
plt.title('Category Representation Across Original Data Sources', fontsize=13, fontweight='bold')
plt.xlabel('Target Category')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.legend(title='Source Dataset')
plt.tight_layout()
plt.show()